- 실습 기본 환경 설정


In [ ]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

# 9-3 어텐션

본 노트북은 본문 9-3절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- 바다나우 어텐션을 계산하는 `Attention` 클래스
- 매 토큰을 생성할 때 동적 콘텍스트 벡터를 만드는 디코더
- 어텐션 적용 여부에 따른 성능 비교([표 9-4])
- 어텐션 가중치 히트맵([그림 9-5], [그림 9-6])

9-2절의 미니배치 Seq2Seq 모델에 어텐션을 더한다. 데이터는 노이즈가 섞인 최대 40자 입력을 사용한다.

## 데이터와 데이터로더

- 9-1절의 데이터 생성 함수에 더해, 입력 날짜 문자열 앞뒤에 무작위 노이즈를 덧붙이는 `generate_noisy_datepairs()`를 사용한다.
- 어휘 사전, 데이터셋, 데이터로더는 9-2절과 같다.

In [ ]:
# 참고 - 데이터 생성(노이즈가 추가된 데이터 사용)
from datetime import datetime, timedelta
import random
import string

src_formats = [
    '%d %B %Y',     # 01 February 2026
    '%d %b %Y',     # 01 Feb 2026
    '%B %d, %Y',    # February 01, 2026
    '%b %d, %Y',    # Feb 01, 2026
    '%m/%d/%Y',     # 02/01/2026
    '%Y/%m/%d',     # 2026/02/01
    '%d-%m-%Y',     # 01-02-2026
    '%Y-%m-%d',     # 2026-02-01
]

# 시작일(start_date)과 종료일(end_date) 사이의 sample_size개의 무작위 날짜 선택
def choice_random_dates(sample_size=1, start_date=None, end_date=None):
    if start_date is None:
        start_date = datetime(1900, 1, 1)
    if end_date is None:
        end_date = datetime(2050, 12, 31)
    days_between = (end_date - start_date).days
    datetime_list = []
    for _ in range(sample_size):
        days_after = random.randrange(days_between)
        datetime_list.append(start_date + timedelta(days=days_after))
    return datetime_list

# datetime 리스트를 입력 리스트와 정답 리스트로 변환하는 함수
# 입력은 src_formats에 정의된 형식으로, 정답은 'YYYY-M-D' 형식으로 변환
def generate_datepairs(date_list, src_formats=src_formats):
    if len(src_formats) == 0:
        raise ValueError('입력 문자열 템플릿이 비어 있습니다.')
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(date.strftime(src_format))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates

# 날짜 문자열 앞뒤에 무작위 길이의 무작위 노이즈를 덧붙인 문자열 생성
def add_random_noise(text, max_length=40):
    noise_chars = (
        string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
    )
    prefix, suffix = '', ''
    remaining = max_length - len(text)
    if remaining > 0:
        prefix_length = random.randint(0, remaining)
        remaining -= prefix_length
        prefix = ''.join(random.choices(noise_chars, k=prefix_length))
    if remaining > 0:
        suffix_length = random.randint(0, remaining)
        suffix = ''.join(random.choices(noise_chars, k=suffix_length))
    return prefix + text + suffix

# 노이즈가 추가된 날짜 문자열 쌍 생성 함수
def generate_noisy_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(add_random_noise(date.strftime(src_format)))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates

# 4,000개의 (노이즈가 섞인 입력, 정답) 날짜 쌍 생성
date_list = choice_random_dates(4000)
src_dates, tgt_dates = generate_noisy_datepairs(date_list)
print('노이즈 데이터셋 예시:')
for i in range(4):
    print(f'  {src_dates[i]!r} -> {tgt_dates[i]!r}')

In [ ]:
# 참고 - 어휘 사전, 데이터셋, 데이터로더(9-2절과 동일)
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SOS_TOKEN, EOS_TOKEN, PAD_TOKEN = '<sos>', '<eos>', '<pad>'
SOS_IDX, EOS_IDX, PAD_IDX = 0, 1, 2
special_tokens = {SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX, PAD_TOKEN: PAD_IDX}

from torch.utils.data import Dataset

# 특수 토큰 조합에 <pad> 추가
SOS_TOKEN, EOS_TOKEN, PAD_TOKEN = '<sos>', '<eos>', '<pad>'
SOS_IDX, EOS_IDX, PAD_IDX = 0, 1, 2
special_tokens = {SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX, PAD_TOKEN: PAD_IDX}

# 어휘 사전 클래스(Vocab)는 9-1절과 동일
class Vocab:
    def __init__(self, sequence_list, special_tokens):
        tokens = set()
        for sequence in sequence_list:
            tokens.update(sequence)
        self.vocab = dict(special_tokens)
        idx_start = len(special_tokens)
        for i, token in enumerate(sorted(tokens)):
            self.vocab[token] = i + idx_start
        self.itos = {idx: token for token, idx in self.vocab.items()}

    def encode(self, input_sequence):
        return [self.vocab[token] for token in input_sequence]

    def decode(self, input_ids):
        return [self.itos[idx] for idx in input_ids]

    def __len__(self):
        return len(self.vocab)

# 데이터셋 클래스: 텐서 변환 제거
class DateDataset(Dataset):
    def __init__(self, src_dates, tgt_dates, src_vocab, tgt_vocab):
        self.samples = []
        for src_date, tgt_date in zip(src_dates, tgt_dates):
            src_ids = src_vocab.encode(src_date)
            tgt_ids = tgt_vocab.encode(tgt_date)
            tgt_ids = [SOS_IDX] + tgt_ids + [EOS_IDX]
            self.samples.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        src_ids, tgt_ids = self.samples[idx]
        # 텐서 변환 없이 정수 리스트를 반환(데이터로더가 텐서로 변환해 모델에 전달)
        return src_ids, tgt_ids

# 배치 병합 함수
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded

# 어휘 사전 생성
src_vocab = Vocab(src_dates, special_tokens)
tgt_vocab = Vocab(tgt_dates, special_tokens)

# 데이터셋(훈련/검증) 생성
train_set = DateDataset(src_dates[:3000], tgt_dates[:3000], src_vocab, tgt_vocab)
valid_set = DateDataset(src_dates[3000:], tgt_dates[3000:], src_vocab, tgt_vocab)

# 배치 크기 32인 데이터로더 생성
BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'입력 어휘 사전 크기: {len(src_vocab)}')
print(f'출력 어휘 사전 크기: {len(tgt_vocab)}')
print(f'훈련/검증 데이터셋: {len(train_set)} / {len(valid_set)}')

## Attention 클래스

- 바다나우 어텐션은 두 개의 선형 계층으로 어텐션 가중치를 계산한다.
    - 어텐션 에너지: 디코더의 현재 상태와 인코더의 각 시점 출력을 각각 선형 변환해 더한 뒤 하이퍼볼릭 탄젠트를 적용한다.
    - 어텐션 점수: 에너지를 또 하나의 선형 계층으로 스칼라 값으로 줄인다.
    - 어텐션 가중치: 점수에 소프트맥스를 적용해 합이 1인 가중치로 만든다.
- 패딩 위치는 가중치가 0이 되도록 소프트맥스 전에 아주 작은 값으로 마스킹한다.

> 본문 [그림 9-4]에서 어텐션 에너지, 점수, 가중치를 계산하는 3단계까지는 `Attention` 클래스가 담당하고, 동적 콘텍스트 벡터를 계산하는 4단계는 디코더가 담당한다.

In [ ]:
######################################################################################
# 코드 9-18 - 어텐션을 계산하는 Attention 클래스
######################################################################################

import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # 어텐션 에너지를 계산하는 선형 계층
        self.attn_energy = nn.Linear(hidden_dim * 2, hidden_dim)
        # 어텐션 점수를 계산하는 선형 계층, 편향 비활성화(bias=False)는 본문 설명 참조
        self.score_projection = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_output, pad_mask=None):
        # decoder_hidden: 디코더 LSTM의 숨겨진 상태 (B, hidden_dim)
        # encoder_output: 인코더 LSTM의 출력 (B, src_length, hidden_dim)
        src_length = encoder_output.shape[1]
        # 현재 생성 맥락(decoder_hidden)과 입력 기억(encoder_output)을 토큰별로 결합
        # 토큰별 입력 기억에 현재 생성 맥락을 연결하기 위해 decoder_hidden을 확장
        hidden_expanded = decoder_hidden.unsqueeze(1).repeat(1, src_length, 1)
                # (B, hidden_dim) -> (B, 1, hidden_dim) -> (B, src_length, hidden_dim)
        combined = torch.cat([hidden_expanded, encoder_output], dim=-1)
                #                 -> (B, src_length, hidden_dim + hidden_dim)

        # ①어텐션 에너지 계산: attn_energy 계층 + Tanh 활성화 함수
        energy = torch.tanh(self.attn_energy(combined))
                #                 -> (B, src_length, hidden_dim)
        # ②어텐션 점수 계산: score_projection 계층 + 스칼라화(squeeze(-1))
        scores = self.score_projection(energy).squeeze(-1)
                #                 -> (B, src_length, 1) -> (B, src_length)
        # ③어텐션 가중치 계산: softmax() 함수 사용
        # <pad> 토큰의 어텐션 점수를 음의 무한대로 바꿈 -> softmax()의 결과는 0
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask == 0, -float('inf'))
        attention_weights = torch.softmax(scores, dim=1)
        return attention_weights

## 어텐션을 적용한 디코더

- 생성자에 어텐션 계층을 추가하고, `forward_step()`에서 매 토큰을 생성할 때마다 어텐션 가중치와 동적 콘텍스트 벡터를 새로 계산한다.
    - 9-1절과 9-2절의 디코더는 고정된 콘텍스트 벡터 하나를 모든 시점에 사용했다.
- 어텐션 가중치는 시각화에 사용하므로 함께 반환한다.

In [ ]:
######################################################################################
# 코드 9-19 - 어텐션 메커니즘을 추가한 디코더 클래스
######################################################################################

# encoder_output: 인코더 LSTM 계층의 모든 시점의 숨겨진 상태(인코더 출력)
# pad_mask: <pad> 위치를 False로 기록한 마스크 텐서
class Decoder(nn.Module):
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        self.attention = Attention(hidden_dim)      # 어텐션 계층 추가
        # 동적 콘텍스트 벡터의 길이: hidden_dim(이전 콘텍스트 벡터와 동일)
        self.decoder_lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    # 토큰 하나를 예측하는 함수
    #   : 고정 콘텍스트 벡터 대신 어텐션 메커니즘의 동적 콘텍스트 벡터를 사용
    def forward_step(self, token, hidden, cell, encoder_output, pad_mask):
        embedded = self.decoder_embedding(token)
        # 인코더 출력(encoder_output)을 사용해 어텐션 가중치 계산
        attention_weights = self.attention(hidden[-1], encoder_output, pad_mask)
        # ④어텐션 가중치를 사용한 가중합으로 동적 콘텍스트 벡터 계산
        #   : 배치 행렬곱 함수 torch.bmm()은 추가 설명 참조
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_output)
        lstm_input = torch.cat([embedded, context], dim=-1)
        output, (hidden, cell) = self.decoder_lstm(lstm_input, (hidden, cell))
        logits = self.fc(output.squeeze(1))
        # 어텐션 가중치도 함께 반환 (시각화용)
        return logits, hidden, cell, attention_weights

    # 순차 데이터 전체 토큰 예측 함수(context 삭제, pad_mask, encoder_output 추가)    
    def forward(self, tgt, hidden, cell, encoder_output, pad_mask=None,
                forcing_ratio=0.5):
        _, target_length = tgt.shape
        all_logits = []
        token = tgt[:, 0:1]
        for i in range(target_length):
            # context 대신 encoder_output과 pad_mask 인자 추가
            logits, hidden, cell, _ = self.forward_step(
                token, hidden, cell, encoder_output, pad_mask
            )
            all_logits.append(logits.unsqueeze(1))
            if i + 1 < target_length:
                use_teacher_forcing = random.random() < forcing_ratio
                if use_teacher_forcing:                         # 교사 강제
                    token = tgt[:, i + 1:i + 2]                 #   : 정답 토큰 사용
                else:                                           # 교사 강제하지 않음
                    token = logits.argmax(dim=-1, keepdim=True) #   : 예측 토큰 사용
        return torch.cat(all_logits, dim=1)

## 어텐션을 적용한 인코더와 DateConverterAttn

- 인코더는 어텐션 계산에 필요한 LSTM 출력(`encoder_output`)을 언패킹해 함께 반환한다.
    - 9-1절과 9-2절의 인코더는 이 값을 버렸다.
- `DateConverterAttn`은 인코더의 출력과 패딩 마스크를 디코더로 전달한다.

In [ ]:
######################################################################################
# 코드 9-20 - 어텐션 메커니즘을 적용한 인코더와 DateConverterAttn 클래스
######################################################################################

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 패킹된 LSTM 출력을 언패킹해서 반환하는 인코더 클래스
class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, 
                                    batch_first=True)

    def forward(self, src, src_lengths):        
        embedded = self.encoder_embedding(src)
        # 패킹 후 LSTM에 입력
        packed_embedding = pack_padded_sequence(embedded, src_lengths.cpu(),
                                                batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.encoder_lstm(packed_embedding)
        # 어텐션 메커니즘에 필요하므로 패킹된 LSTM 출력을 언패킹해서 함께 반환
        encoder_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        return encoder_output, hidden, cell


# 어텐션 메커니즘을 적용한 Seq2Seq 날짜 변환기 클래스
class DateConverterAttn(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, src_lengths, forcing_ratio=0.5):
        # encoder_output까지 반환값에 추가
        encoder_output, hidden, cell = self.encoder(src, src_lengths)
        # <pad> 토큰 위치를 False로 기록한 패딩 마스크 생성
        pad_mask = (src != PAD_IDX)              # (B, src_length)
        tgt_input = tgt[:, :-1]
        # 콘텍스트 벡터 대신 encoder_output, pad_mask를 디코더에 전달
        logits = self.decoder(tgt_input, hidden, cell, encoder_output, pad_mask, forcing_ratio)
        return logits

## 모델의 학습([표 9-4])

- 어텐션은 클래스 내부에만 적용되므로 9-2절의 학습 함수를 그대로 사용한다.

In [ ]:
# 참고 - 학습 함수(9-2절과 동일)

import copy

def train_epoch(model, train_loader, criterion, optimizer, device=None):
    if device is None: device = torch.device('cpu')
    model.train()
    total_loss, token_count = 0.0, 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        optimizer.zero_grad()
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
    return total_loss / token_count

@torch.no_grad()
def validation(model, valid_loader, criterion, device=None):
    if device is None:
        device = torch.device('cpu')
    model.eval()
    total_loss, token_count = 0.0, 0
    correct_size, sample_size = 0, 0
    for src, tgt in valid_loader:
        src, tgt = src.to(device), tgt.to(device)
        # 검증에서도 길이 정보 텐서 생성 및 전달이 필요
        src_lengths = (src != PAD_IDX).sum(dim=1)
        logits = model(src, tgt, src_lengths, forcing_ratio=0.0)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
        preds = logits.argmax(dim=-1)
        mask = labels != PAD_IDX
        # <pad> 위치를 제외하고 샘플 별 정답/오답을 판정해 취합
        sample_correct = ((preds == labels) | ~mask).all(dim=1)
        correct_size += sample_correct.sum().item()
        sample_size += src.size(0)
    return total_loss / token_count, correct_size / sample_size * 100.0

def train_with_early_stopping(model, train_loader, valid_loader, criterion, optimizer, 
                              epochs, patience, device=None):
    if device is None: 
        device = torch.device('cpu')
    model.to(device)
    log = common.EpochLogger(epochs, target_rows=epochs)
    best_valid_loss = float('inf')
    best_state, counter = None, 0
    stopped = False
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = validation(model, valid_loader, criterion, device)        
        log.row(epoch, train_loss, valid_loss, valid_acc)
        if valid_loss < best_valid_loss:            
            best_valid_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:                                       
            counter += 1
            if counter >= patience:                 
                stopped = True
                break
    if best_state is not None:
        model.load_state_dict(best_state)           
    log.summary(stopped='조기 종료' if stopped else None)

    return log    

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# 참고 - 어텐션 모델 생성과 학습

import torch.optim as optim

# 하이퍼파라미터 설정
EMBED_DIM = 32
HIDDEN_DIM = 32
NUM_LAYERS = 1
LR = 1e-3
EPOCHS = 200
PATIENCE = 5

# 결과 재현을 위한 시드값 설정
common.set_seed(SEED)

# 모델 생성 방법은 어텐션이 없는 기존 Seq2Seq 모델과 동일
encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
model = DateConverterAttn(encoder, decoder).to(device)

# 옵티마이저, 손실 함수 생성
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

log = train_with_early_stopping(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=EPOCHS, patience=PATIENCE, device=device
)

In [ ]:
# 참고 - 어텐션 모델 학습 곡선
log.plot(title='DateConverterAttn 학습 곡선 (노이즈 데이터셋)')

In [ ]:
# 참고 - 어텐션 모델의 파라미터 수
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

param_count = count_parameters(model)
print(f'어텐션 모델 파라미터 수: {param_count:,}개')

- 본문 [표 9-4]의 수치와 비교해 보자.

| 모델 | 최적 에포크 | 학습 시간(초) | 훈련 손실 | 검증 손실 | 검증 정확도(%) |
|---|---|---|---|---|---|
| 어텐션 미적용 | 77 | 144 (1.8) | 0.1267 | 0.2904 | 55.00 |
| 어텐션 적용 | 63 | 216 (3.2) | 0.0125 | 0.0679 | 90.10 |

- 어텐션 미적용 모델의 학습 과정은 9-2절 노트북(`09-02_example.ipynb`)의 마지막 참고 셀에 있다.

## 변환 결과 확인

- 예측 함수는 한 샘플만 예측해 패딩이 없더라도, 디코더의 시그니처에 맞춰 패딩 마스크를 만들어 `encoder_output`과 함께 전달해야 한다.
- 시각화를 위해 어텐션 가중치도 함께 수집한다.

In [ ]:
# 참고 - 어텐션 모델의 예측 함수(시각화를 위한 어텐션 가중치도 함께 수집)
import numpy as np

@torch.no_grad()
def predict_with_attention(model, src_text, src_vocab, tgt_vocab, device=None, max_length=12):
    if device is None: device = torch.device('cpu')
    model.eval()

    src_ids = src_vocab.encode(src_text)
    src = torch.LongTensor(src_ids).unsqueeze(0).to(device)
    # 인코더 - 동적 컨텍스트 벡터를 만들기 위한 숨겨진 상태(encoder_output)뿐 아니라 
    #          디코더 LSTM 초기화를 위한 마지막 숨겨진 상태(hidden)와 셀 상태(cell)도 함께 반환
    src_lengths = torch.LongTensor([len(src_ids)])
    encoder_output, hidden, cell = model.encoder(src, src_lengths)
    # 샘플 하나만 예측해도 패딩 마스크 필요(디코더 입력 인자)
    pad_mask = (src != PAD_IDX)
    tgt_token = torch.tensor([[SOS_IDX]]).to(device)
    result_ids = []
    attention_list = []
    # 최대 max_length 토큰까지 자기 회귀 방식으로 반복 생성
    for _ in range(max_length):
        logits, hidden, cell, attn = model.decoder.forward_step(
            tgt_token, hidden, cell, encoder_output, pad_mask
        )
        prediction = logits.argmax(dim=-1, keepdim=True)
        # 어텐션 가중치 수집(넘파이 배열로 변환)
        attention_list.append(attn[0].cpu().numpy())
        if prediction.item() == EOS_IDX:
            break
        result_ids.append(prediction.item())
        tgt_token = prediction
    text = ''.join(tgt_vocab.decode(result_ids))

    # 출력 토큰 수만큼만 어텐션 가중치 병합
    attention_weights = np.stack(attention_list[:len(text)])
    return text, attention_weights


def predict(model, src_text, src_vocab, tgt_vocab, device=None, max_length=12):
    text, _ = predict_with_attention(model, src_text, src_vocab, tgt_vocab, device, max_length)
    return text

In [ ]:
# 참고 - 노이즈가 없을 때와 있을 때의 변환 결과
# 참고 - 임의 날짜에 대해서 노이지가 없을 때의 있을 때의 변환 결과

common.set_seed(SEED)
test_date = choice_random_dates(1) * 8
clean_src, clean_tgt = generate_datepairs(test_date)
noisy_src, noisy_tgt = generate_noisy_datepairs(test_date)
print(f'정답 날짜 문자열: {clean_tgt[0]}\n')
print('노이즈 없는 날짜 문자열 변환:')
print('  입력 날짜 문자열                         => 예측 날짜 문자열 (정답 여부)')
for src_text, tgt_text in zip(clean_src, clean_tgt):
    pred = predict(model, src_text, src_vocab, tgt_vocab, device)
    mark = '정답' if pred == tgt_text else '오답'
    print(f'  {src_text:<40s} => {pred} ({mark})')

print('\n노이즈를 추가한 날짜 문자열 변환:')
print('  입력 날짜 문자열                         => 예측 날짜 문자열 (정답 여부)')
for src_text, tgt_text in zip(noisy_src, noisy_tgt):
    pred = predict(model, src_text, src_vocab, tgt_vocab, device)
    mark = '정답' if pred == tgt_text else '오답'
    print(f'  {src_text:<40s} => {pred} ({mark})')

## 어텐션 가중치 시각화([그림 9-5], [그림 9-6])

- 어텐션 가중치 히트맵으로 모델이 각 출력 토큰을 생성할 때 입력의 어느 위치에 집중했는지 확인할 수 있다.
    - 가로축이 입력 토큰, 세로축이 생성한 출력 토큰이다. 밝을수록 가중치가 크다.

In [ ]:
# 참고 - 'Sep 18, 1938'을 입력했을 때의 어텐션 가중치 히트맵([그림 9-5])
src_text = 'Sep 18, 1938'
pred, attention_weights = predict_with_attention(model, src_text, src_vocab, tgt_vocab, device)
print(f'{src_text!r} → {pred!r}')

# 시각화
viz.plot_attention(
    attention_weights,
    source_tokens=list(src_text),
    target_tokens=list(pred),
    title=f'{src_text!r} → {pred!r}',
    figsize=(4, 4),
    gamma=0.3
)

In [ ]:
# 참고 - '18 Sep 1938'을 입력했을 때의 어텐션 가중치 히트맵([그림 9-5])

src_text = '18 Sep 1938'
pred, attention_weights = predict_with_attention(model, src_text, src_vocab, tgt_vocab, device)
print(f'{src_text!r} → {pred!r}')

# 시각화
viz.plot_attention(
    attention_weights,
    source_tokens=list(src_text),
    target_tokens=list(pred),
    title=f'{src_text!r} → {pred!r}',
    figsize=(4, 4),
    gamma=0.3
)

- 입력 토큰의 위치와 무관하게 `1938`을 생성할 때는 입력에서 연도에 해당하는 부분에 집중한다.
    - 입력 형식이 달라도 필요한 정보를 찾아내는 것이 어텐션의 힘이다.

In [ ]:
# 참고 - 긴 문자열 날짜('18 September 1938')를 입력했을 때의 어텐션 가중치 히트맵 

src_text = '18 September 1938'
pred, attention_weights = predict_with_attention(model, src_text, src_vocab, tgt_vocab, device)
print(f'{src_text!r} → {pred!r}')

# 시각화
viz.plot_attention(
    attention_weights,
    source_tokens=list(src_text),
    target_tokens=list(pred),
    title=f'{src_text!r} → {pred!r}',
    figsize=(8, 4),
    gamma=0.3
)

In [ ]:
# 참고 - 노이즈가 섞인 문자열 날짜를 입력했을 때의 어텐션 가중치 히트맵([그림 9-6])

# 입력, 정답 문자열
src_text = '0q?-U:k!U]}T1991/07/15FR?OdR{z=W@@b'
date_str = '1991/07/15'

start = src_text.index(date_str)
src_labels = [''] * len(src_text)
src_labels[start:start + len(date_str)] = list(date_str)
pred, attention_weights = predict_with_attention(model, src_text, src_vocab, tgt_vocab, device)
print(f'{src_text!r} → {pred!r}')

# 시각화
viz.plot_attention(
    attention_weights,
    source_tokens=src_labels,
    target_tokens=list(pred),
    title=f'{src_text!r} → {pred!r}',
    figsize=(8, 4),
    gamma=0.3
)

## 정리

- 어텐션 메커니즘은 매 출력 토큰을 생성할 때 입력의 어느 부분에 집중할지 동적으로 계산해 동적 콘텍스트 벡터를 만든다.
- 덕분에 노이즈가 섞인 긴 입력에서도 필요한 정보를 찾아내, 어텐션이 없는 모델의 정보 병목 문제를 크게 줄인다.
- 어텐션 가중치를 히트맵으로 시각화하면 모델이 무엇을 보고 판단했는지 눈으로 확인할 수 있다.